In [ ]:
# # 00 — Full-history data audit

# ## Objective

# Audit every original EURUSD tick CSV, preserve the source observations,
# and produce evidence for the ETL design discussion.

# ## Prerequisites

# - Run in the existing Jupyter Python environment.
# - The repository contains data/ and notebooks/.
# - Original CSVs are directly inside data/.
# - Required libraries: pandas, NumPy, PyArrow, DuckDB, Matplotlib.

# ## Inputs

# Expected CSV columns:

# Time (EET), Ask, Bid, AskVolume, BidVolume

# Timestamp interpretation: Europe/Athens.
# Expected timestamp format: YYYY.MM.DD HH:MM:SS.fff.
# Prices: expected USD per EUR; one pip = 0.0001.
# Size denomination remains unconfirmed. Displayed sizes are not trade volumes.

# ## Outputs

# - README.md, CONTRIBUTIONS.md, .gitignore.
# - Shared functions under functions/data_audit/.
# - Calendar definitions and design records under docs/.
# - An isolated audit run under reports/tables/data_audit/.
# - A frequency-distribution figure under reports/figures/data_audit/.

# ## Audit conventions

# - Read every original CSV without a historical date filter.
# - Preserve original strings and source-record references.
# - Do not sort, deduplicate, filter quotes, or fill gaps.
# - Use actual elapsed time with timezone-aware timestamps.
# - Carry the preceding record across batches and files.
# - Count invalid timestamps and unresolved clock times explicitly.
# - Attribute an interval to the Athens year of its ending quote.
# - Record same-timestamp updates and unchanged-price observations separately.

# The duplicate diagnostics concern adjacent source records. They do not
# establish global raw-row uniqueness.

# Holiday dates are context annotations, not automatic quote closures.
# EUR annotations use the Brussels civil date; USD annotations use the
# New York civil date. Their bounds are converted to Athens time.
# 24 and 31 December are additional Athens-date context annotations.

# Weekend closure baseline:
# Friday 17:00 New York to Sunday 17:00 New York.

# Exact, independently verified quote-closure intervals can be supplied
# separately. Public holidays and settlement holidays do not populate
# that list automatically.

# No disruption threshold is selected in this notebook.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata, util
import csv
import importlib
import json
import platform
import re
import shutil
import subprocess
import sys

candidates = [Path.cwd(), *Path.cwd().parents]

REPO_ROOT = next(
    (
        p.resolve()
        for p in candidates
        if (p / "data").is_dir() and (p / "notebooks").is_dir()
    ),
    None,
)

if REPO_ROOT is None:
    REPO_ROOT = Path(
        input("Full repository path: ").strip().strip('"')
    ).expanduser().resolve()

assert (REPO_ROOT / "data").is_dir(), "data/ was not found."
assert (REPO_ROOT / "notebooks").is_dir(), "notebooks/ was not found."

required = ["numpy", "pandas", "pyarrow", "duckdb", "matplotlib"]
missing = [name for name in required if util.find_spec(name) is None]

print("Repository:", REPO_ROOT)
print("Python:", platform.python_version())
print("Active interpreter:", sys.executable)
print("Virtual environment:", sys.prefix != sys.base_prefix)
print("Free disk space, GiB:", round(shutil.disk_usage(REPO_ROOT).free / 2**30, 2))

if missing:
    print("\nRun this in a separate Jupyter cell:")
    print("%pip install " + " ".join(missing))
    raise RuntimeError(
        "Install the missing libraries, restart the kernel, "
        "and rerun from Cell 2."
    )

ENVIRONMENT = {
    "python": platform.python_version(),
    **{name: metadata.version(name) for name in required},
}

print(json.dumps(ENVIRONMENT, indent=2))

In [ ]:
for relative in [
    "docs",
    "functions/data_audit",
    "reports/tables/data_audit",
    "reports/figures/data_audit",
]:
    (REPO_ROOT / relative).mkdir(parents=True, exist_ok=True)

ignore_patterns = [
    "data/",
    "outputs/",
    "reports/tables/",
    "reports/figures/",
    ".local/",
    "work/",
    ".venv/",
    "venv/",
    ".env",
    ".env.*",
    ".ipynb_checkpoints/",
    "__pycache__/",
    "*.py[cod]",
    "*.duckdb",
    "*.duckdb.wal",
    "*.log",
    ".DS_Store",
    "Thumbs.db",
    "desktop.ini",
]

ignore_path = REPO_ROOT / ".gitignore"
existing = (
    ignore_path.read_text(encoding="utf-8")
    if ignore_path.exists()
    else ""
)

existing_lines = set(existing.splitlines())
additions = [p for p in ignore_patterns if p not in existing_lines]

if additions:
    separator = "\n" if existing and not existing.endswith("\n") else ""
    ignore_path.write_text(
        existing
        + separator
        + "\n# Research data, generated files, and local configuration\n"
        + "\n".join(additions)
        + "\n",
        encoding="utf-8",
    )

print(".gitignore ready.")

In [ ]:
git = shutil.which("git")
GIT_STATUS = {"checked": False, "untracked_from_index": 0}

if git is None:
    print("Git executable not found. Return this result with the audit summary.")
else:
    result = subprocess.run(
        [git, "-C", str(REPO_ROOT), "rev-parse", "--show-toplevel"],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            "The selected folder is not recognized as a Git repository. "
            "Return this output before changing the Git setup."
        )

    git_root = Path(result.stdout.strip()).resolve()
    assert git_root == REPO_ROOT, "Git root differs from the selected repository."

    protected = [
        "data",
        "outputs",
        "reports/tables",
        "reports/figures",
        ".env",
        ".env.*",
    ]

    tracked = subprocess.run(
        [git, "-C", str(REPO_ROOT), "ls-files", "-z", "--", *protected],
        check=True,
        capture_output=True,
    ).stdout

    tracked_files = [
        item.decode("utf-8", errors="replace")
        for item in tracked.split(b"\0")
        if item
    ]

    if tracked_files:
        print("Removing protected files from the Git index:", len(tracked_files))
        subprocess.run(
            [
                git, "-C", str(REPO_ROOT),
                "rm", "-r", "--cached", "--ignore-unmatch",
                "--", *protected,
            ],
            check=True,
            capture_output=True,
            text=True,
        )

    remaining = subprocess.run(
        [git, "-C", str(REPO_ROOT), "ls-files", "-z", "--", *protected],
        check=True,
        capture_output=True,
    ).stdout

    assert not remaining, "Protected files remain tracked."

    GIT_STATUS = {
        "checked": True,
        "untracked_from_index": len(tracked_files),
    }

    print("PASS: protected paths are absent from the current Git index.")
    print("Existing commit history has not been changed.")

In [ ]:
#initialize documentation#

readme_text = """# Inventory-Aware FX Pricing and Hedging Simulator

A notebook-first research simulator investigating how market conditions,
forecasts, synthetic client flow, pricing, inventory, hedging, and risk exits
interact during historical EURUSD market replay.

## Data

Original Dukascopy/JForex EURUSD Level 1 tick exports are held locally in
data/. Actual coverage, quality, timestamp conventions, and size
interpretation are subject to audit.

Research timestamps use Europe/Athens with actual elapsed-time calculations.
Displayed sizes are not executed trade volumes.
One feed does not represent the entire FX market.

## Development workflow

Discuss → specify → build → execute → review.

Raymond executes the notebooks. Important choices are agreed immediately
before the relevant component is built.

## Notebook order

00_data_audit.ipynb
01_etl_and_clean_quotes.ipynb
02_sampling_candles_and_indicators.ipynb
03_multiscale_volatility.ipynb
04_entropy.ipynb
05_targets_and_validation_design.ipynb
06_gradient_boosted_regression.ipynb
07_market_replay_and_accounting.ipynb
08_client_flow_and_pricing.ipynb
09_position_accumulation.ipynb
10_risk_exits_and_hedging.ipynb
11_simulation_and_evaluation.ipynb
12_results_and_documentation.ipynb

## Current methodology

The first component is a full-history, bounded-memory audit.
Original rows are preserved. Quote quality, repeated observations,
ordering, coverage, and quote-arrival intervals are measured separately.

Public holidays, settlement holidays, and quote-feed closures are distinct.
No gap threshold, forecasting target, model, pricing policy, accumulation
rule, or hedging rule has yet been selected.

## Environment

Use the existing Jupyter Python environment.
Initial libraries: pandas, NumPy, PyArrow, DuckDB, Matplotlib.
Each audit run records the actual versions used.

Clear saved notebook outputs before committing if they contain local
paths or raw-data examples.

## Results

Results pending execution and validation.

## Limitations

This is a research simulator.
Client behavior is synthetic unless actual client data is supplied.
Predictive accuracy does not establish trading profitability.
Simulated profitability depends on the modeling assumptions.

## Development approach

Development approach: This is an AI-assisted research project.
Raymond Fendi defined the business objectives and provided FX domain
direction. OpenAI Codex assisted with implementation, validation code,
and documentation. CONTRIBUTIONS.md records completed design decisions,
reviews, and validation.
"""

contributions_text = """# Contributions

## Agreed design

Raymond Fendi defined the research objectives and supplied FX domain direction.

The agreed initial implementation is notebook-first, uses Europe/Athens,
preserves original observations, and audits the complete available history.

OpenAI Codex prepared the implementation, inline validation code, and
documentation for Notebook 00.

## Execution and review status

Execution and human review are pending.
No research result or user validation is claimed.
"""

for filename, content in [
    ("README.md", readme_text),
    ("CONTRIBUTIONS.md", contributions_text),
]:
    path = REPO_ROOT / filename
    if path.exists():
        print("Retained existing:", filename)
    else:
        path.write_text(content, encoding="utf-8")
        print("Created:", filename)

In [ ]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
import matplotlib.pyplot as plt
from IPython.display import display

TZ = "Europe/Athens"
PIP_SIZE = 0.0001
CHUNK_ROWS = 100_000
EXAMPLES_PER_CASE_PER_YEAR = 20

RAW_DIR = REPO_ROOT / "data"
FUNCTION_DIR = REPO_ROOT / "functions/data_audit"

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR = REPO_ROOT / "reports/tables/data_audit" / RUN_ID
FIGURE_DIR = REPO_ROOT / "reports/figures/data_audit" / RUN_ID

RUN_DIR.mkdir(parents=True)
FIGURE_DIR.mkdir(parents=True)

FIELDS = ["Time (EET)", "Ask", "Bid", "AskVolume", "BidVolume"]

all_data_files = sorted(p for p in RAW_DIR.rglob("*") if p.is_file())
raw_files = [
    p for p in all_data_files
    if p.parent == RAW_DIR and p.suffix.lower() == ".csv"
]

harmless_names = {".gitkeep", ".DS_Store", "desktop.ini", "Thumbs.db"}
unclassified = [
    p.relative_to(REPO_ROOT).as_posix()
    for p in all_data_files
    if p not in raw_files and p.name not in harmless_names
]

if unclassified:
    display(pd.DataFrame({"unclassified_file": unclassified}))
    raise RuntimeError(
        "Additional data files require classification. "
        "The audit will not silently omit them or combine possible copies "
        "with the original exports."
    )

if not raw_files:
    raise FileNotFoundError("No original CSVs found directly inside data/.")

inventory_records = []
preview_records = []
expected_metadata = {}

for path in raw_files:
    stat = path.stat()
    relative = path.relative_to(REPO_ROOT).as_posix()

    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle)
        header = next(reader, None)

        if header != FIELDS:
            raise ValueError(f"Unexpected header in {relative}: {header}")

        for number, row in zip(range(1, 6), reader):
            preview_records.append({
                "source_file": relative,
                "source_record": number,
                "raw_fields": repr(row),
                "field_count": len(row),
            })

    inventory_records.append({
        "source_file": relative,
        "bytes": stat.st_size,
        "mtime_ns": stat.st_mtime_ns,
    })
    expected_metadata[relative] = (stat.st_size, stat.st_mtime_ns)

inventory = pd.DataFrame(inventory_records)
previews = pd.DataFrame(preview_records)

inventory.to_csv(RUN_DIR / "input_inventory.csv", index=False)
previews.to_csv(RUN_DIR / "input_previews.csv", index=False)
(RUN_DIR / "environment.json").write_text(
    json.dumps(ENVIRONMENT, indent=2), encoding="utf-8"
)

name_years = [
    int(value)
    for path in raw_files
    for value in re.findall(r"(?<!\d)(?:19|20)\d{2}(?!\d)", path.name)
]

if not name_years:
    raise ValueError("Cannot establish a calendar envelope from filenames.")

FIRST_NOMINAL_YEAR = min(name_years)
LAST_NOMINAL_YEAR = max(name_years)

print("Mode: FULL HISTORY — no date filter")
print("Original CSV files:", len(raw_files))
print("Source bytes:", int(inventory["bytes"].sum()))
print("Source GiB:", round(inventory["bytes"].sum() / 2**30, 3))
print("Run directory:", RUN_DIR)

display(inventory.drop(columns="mtime_ns"))
display(previews)

In [ ]:
# ## Calendar corrections and gap populations

# The calendar distinguishes:

# - EUR TARGET2/T2 settlement holidays.
# - U.S. actual holiday dates.
# - U.S. federal observed dates.
# - Federal Reserve payment-system holiday dates.
# - Research context for 24 and 31 December.

# Corrections include:

# - Good Friday is not a standard U.S. federal/payment holiday.
# - Saturday holidays do not close Federal Reserve Banks on Friday.
# - Sunday holidays close Federal Reserve Banks on Monday.
# - New Year 2022 was publicly observed on 31 December 2021,
#   not 3 January 2022.
# - Juneteenth is introduced from 2021; its 2021 federal observation
#   does not create a Friday payment-service closure.
# - EUR weekend holidays do not create substitute Monday TARGET closures.

# Calendar dates are retrospective context labels.

# The main gap population is called `unannotated_open`, meaning:
# no overlap with the selected closure baseline or holiday annotations.
# It is not proof that every possible liquidity-related holiday is covered.

# Other populations:

# - `holiday_context`
# - `closure_crossing`
# - `calendar_uncovered`

# For closure-crossing gaps, open portions before, after, and between
# closures are retained separately. These portions are censored intervals:
# a calendar boundary is not an observed quote.

# The audit saves every positive interval longer than 10 seconds, plus
# all closure-crossing and calendar-uncovered positive intervals.
# This recording boundary is not the disruption threshold.

In [ ]:
helper_sources = {}

helper_sources["fx_audit_calendar.py"] = r'''# Generated by Notebook 00
from datetime import date, timedelta
import numpy as np
import pandas as pd

TZ = "Europe/Athens"
ECB = "https://www.ecb.europa.eu/press/pr/date/2000/html/pr001214_4.en.html"
OPM = "https://www.opm.gov/policy-data-oversight/pay-leave/federal-holidays/"
FED = "https://www.frbservices.org/about/holiday-schedules/"
JUNETEENTH = "https://www.federalreserve.gov/newsevents/pressreleases/other20210617b.htm"
SESSIONS = "https://www.dukascopy.com/wiki/en/manuals/jforex4-desktop/trading-sessions-logic/"

def easter(year):
    a = year % 19
    b, c = divmod(year, 100)
    d, e = divmod(b, 4)
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19*a + b - d - g + 15) % 30
    i, k = divmod(c, 4)
    l = (32 + 2*e + 2*i - h - k) % 7
    m = (a + 11*h + 22*l) // 451
    value = h + l - 7*m + 114
    return date(year, value // 31, value % 31 + 1)

def nth_weekday(year, month, weekday, number):
    first = date(year, month, 1)
    return first + timedelta(
        days=(weekday - first.weekday()) % 7 + 7*(number - 1)
    )

def last_monday_may(year):
    day = date(year, 5, 31)
    return day - timedelta(days=day.weekday())

def day_window(day, zone):
    start = pd.Timestamp(day).tz_localize(zone)
    end = pd.Timestamp(day + timedelta(days=1)).tz_localize(zone)
    return start.tz_convert(TZ).value, end.tz_convert(TZ).value

def merge_windows(windows):
    result = []
    for start, end in sorted(windows):
        if end <= start:
            raise ValueError("Closure/context end must follow its start.")
        if result and start <= result[-1][1]:
            result[-1] = (result[-1][0], max(result[-1][1], end))
        else:
            result.append((start, end))
    return result

def overlap_parts(a, b, windows):
    n = len(a)
    total = np.zeros(n, dtype=np.int64)
    left = np.zeros(n, dtype=np.int64)
    right = np.zeros(n, dtype=np.int64)
    middle = np.zeros(n, dtype=np.int64)
    if not n:
        return total, left, right, middle

    first = np.full(n, np.iinfo(np.int64).max, dtype=np.int64)
    last = np.full(n, np.iinfo(np.int64).min, dtype=np.int64)
    lower, upper = int(a.min()), int(b.max())

    for start, end in windows:
        if end <= lower:
            continue
        if start >= upper:
            break
        amount = np.maximum(
            0, np.minimum(b, end) - np.maximum(a, start)
        )
        hit = amount > 0
        total += amount
        first[hit] = np.minimum(first[hit], start)
        last[hit] = np.maximum(last[hit], end)

    hit = total > 0
    left[hit] = np.maximum(0, np.minimum(b[hit], first[hit]) - a[hit])
    right[hit] = np.maximum(0, b[hit] - np.maximum(a[hit], last[hit]))
    middle[hit] = b[hit] - a[hit] - total[hit] - left[hit] - right[hit]

    assert np.all(total <= b - a)
    assert np.all(middle >= 0)
    return total, left, right, middle

def points_inside(values, valid, windows):
    result = np.zeros(len(values), dtype=bool)
    if not valid.any():
        return result
    lower, upper = int(values[valid].min()), int(values[valid].max())
    for start, end in windows:
        if end <= lower:
            continue
        if start > upper:
            break
        result |= valid & (values >= start) & (values < end)
    return result

def build_calendar(first_year, last_year, verified_closures):
    events, context = [], []

    def add(day, name, kind, currency, zone, nominal_year, source):
        start, end = day_window(day, zone)
        events.append({
            "date": day.isoformat(),
            "holiday": name,
            "kind": kind,
            "currency": currency,
            "calendar_zone": zone,
            "nominal_year": nominal_year,
            "weekend": day.weekday() >= 5,
            "source_url": source,
        })
        context.append((start, end))

    for year in range(first_year - 1, last_year + 2):
        sunday = easter(year)
        eur = [
            (date(year, 1, 1), "New Year's Day"),
            (sunday - timedelta(days=2), "Good Friday"),
            (sunday + timedelta(days=1), "Easter Monday"),
            (date(year, 5, 1), "Labour Day"),
            (date(year, 12, 25), "Christmas Day"),
            (date(year, 12, 26), "Boxing Day"),
        ]
        for day, name in eur:
            add(day, name, "eur_settlement", "EUR",
                "Europe/Brussels", year, ECB)

        us = [
            (date(year, 1, 1), "New Year's Day"),
            (nth_weekday(year, 1, 0, 3), "Martin Luther King Jr. Day"),
            (nth_weekday(year, 2, 0, 3), "Washington's Birthday"),
            (last_monday_may(year), "Memorial Day"),
            (date(year, 7, 4), "Independence Day"),
            (nth_weekday(year, 9, 0, 1), "Labor Day"),
            (nth_weekday(year, 10, 0, 2), "Columbus Day"),
            (date(year, 11, 11), "Veterans Day"),
            (nth_weekday(year, 11, 3, 4), "Thanksgiving Day"),
            (date(year, 12, 25), "Christmas Day"),
        ]
        if year >= 2021:
            us.append((date(year, 6, 19), "Juneteenth"))

        for actual, name in us:
            observed = actual
            if actual.weekday() == 5:
                observed -= timedelta(days=1)
            elif actual.weekday() == 6:
                observed += timedelta(days=1)

            payment = (
                actual + timedelta(days=1)
                if actual.weekday() == 6 else actual
            )
            add(actual, name, "us_actual", "USD",
                "America/New_York", year, OPM)
            add(observed, name, "us_public_observed", "USD",
                "America/New_York", year, OPM)
            add(payment, name, "us_payment", "USD",
                "America/New_York", year,
                JUNETEENTH if name == "Juneteenth" and year == 2021 else FED)

        for day in [date(year, 12, 24), date(year, 12, 31)]:
            add(day, "Open holiday-affected session", "research_context",
                "EURUSD", TZ, year, "Agreed project convention")

    begin = pd.Timestamp(f"{first_year - 1}-01-01", tz=TZ)
    end = pd.Timestamp(f"{last_year + 2}-01-01", tz=TZ)
    closures, closure_rows = [], []

    fridays = pd.date_range(
        f"{first_year - 1}-01-01",
        f"{last_year + 2}-01-01",
        freq="W-FRI",
    )

    for friday in fridays:
        # Localize both endpoints independently across DST weekends.
        start = friday.replace(hour=17).tz_localize("America/New_York")
        finish = (
            friday + pd.Timedelta(days=2)
        ).replace(hour=17).tz_localize("America/New_York")

        closures.append((start.value, finish.value))
        closure_rows.append({
            "start": start.tz_convert(TZ).isoformat(),
            "end": finish.tz_convert(TZ).isoformat(),
            "kind": "expected_weekend",
            "source_url": SESSIONS,
        })

    for row in verified_closures.to_dict("records"):
        start, finish = pd.Timestamp(row["start"]), pd.Timestamp(row["end"])
        if start.tzinfo is None or finish.tzinfo is None:
            raise ValueError("Verified closure timestamps require UTC offsets.")
        if not str(row["source_url"]).strip():
            raise ValueError("Verified quote closures require a source URL.")
        closures.append((start.value, finish.value))
        closure_rows.append({
            "start": start.tz_convert(TZ).isoformat(),
            "end": finish.tz_convert(TZ).isoformat(),
            "kind": "verified_quote_closure",
            "source_url": row["source_url"],
        })

    return (
        pd.DataFrame(events),
        pd.DataFrame(closure_rows),
        merge_windows(closures),
        merge_windows(context),
        (begin.value, end.value),
    )
'''

In [ ]:
helper_sources["fx_audit_rows.py"] = r'''# Generated by Notebook 00
import numpy as np
import pandas as pd
from fx_audit_calendar import overlap_parts, points_inside

TZ = "Europe/Athens"
FIELDS = ["Time (EET)", "Ask", "Bid", "AskVolume", "BidVolume"]
VALUES = FIELDS[1:]
PIP_SIZE = 0.0001

UPPER_SECONDS = np.array([
    1, 2, 5, 10,
    *range(15, 76, 5),
    *range(105, 586, 30),
    600,
    *range(660, 1801, 60),
], dtype=np.int64)

UPPER_NS = UPPER_SECONDS * 1_000_000_000
LOWER_SECONDS = np.r_[0, UPPER_SECONDS]
LABELS = [
    f"({lower},{upper}]"
    for lower, upper in zip(np.r_[0, UPPER_SECONDS[:-1]], UPPER_SECONDS)
] + [">1800"]

def parse_athens(text):
    parsed = pd.to_datetime(
        text.str.strip(),
        format="%Y.%m.%d %H:%M:%S.%f",
        errors="coerce",
        exact=True,
    )
    # A technical guard for nanosecond arithmetic, not a data filter.
    supported = parsed.between(
        pd.Timestamp("1900-01-01"),
        pd.Timestamp("2100-01-01"),
        inclusive="left",
    )
    outside = parsed.notna() & ~supported
    failed = parsed.isna()
    local = parsed.where(supported).astype("datetime64[ns]")
    aware = local.dt.tz_localize(
        TZ, ambiguous="NaT", nonexistent="NaT"
    )
    unresolved = local.notna() & aware.isna()
    return aware, failed, outside, unresolved

def ns_values(times):
    return (
        times.dt.tz_convert("UTC")
        .dt.tz_localize(None)
        .to_numpy(dtype="datetime64[ns]")
        .view("i8")
    )

def audit_rows(raw, previous, closures, context, bounds):
    joined = (
        pd.concat([previous, raw], ignore_index=True)
        if previous is not None else raw.reset_index(drop=True)
    )
    offset = 1 if previous is not None else 0
    d = joined.copy()

    for column in [*FIELDS, "source_file", "source_record"]:
        d["previous_" + column] = joined[column].shift(1)

    times, failed, outside, unresolved = parse_athens(joined[FIELDS[0]])
    prior_times = times.shift(1)
    current_ns, prior_ns = ns_values(times), ns_values(prior_times)

    has_previous = np.arange(len(d)) > 0
    valid_pair = (
        has_previous & times.notna().to_numpy()
        & prior_times.notna().to_numpy()
    )

    delta_ns = np.zeros(len(d), dtype=np.int64)
    delta_ns[valid_pair] = (
        current_ns[valid_pair] - prior_ns[valid_pair]
    )

    positive = valid_pair & (delta_ns > 0)
    equal = valid_pair & (delta_ns == 0)
    backward = valid_pair & (delta_ns < 0)

    numbers = pd.DataFrame({
        c: pd.to_numeric(joined[c].str.strip(), errors="coerce")
        .to_numpy(dtype="float64", na_value=np.nan)
        for c in VALUES
    })

    good_price = (
        np.isfinite(numbers["Ask"]) & np.isfinite(numbers["Bid"])
        & numbers["Ask"].gt(0) & numbers["Bid"].gt(0)
    )
    good_size = (
        np.isfinite(numbers["AskVolume"])
        & np.isfinite(numbers["BidVolume"])
        & numbers["AskVolume"].ge(0) & numbers["BidVolume"].ge(0)
    )

    exact_raw = (
        joined[FIELDS].eq(joined[FIELDS].shift(1))
        .fillna(False).all(axis=1)
    )
    same_raw_quote = (
        joined[VALUES].eq(joined[VALUES].shift(1))
        .fillna(False).all(axis=1)
    )
    same_prices = (
        good_price & good_price.shift(1, fill_value=False)
        & numbers["Ask"].eq(numbers["Ask"].shift(1))
        & numbers["Bid"].eq(numbers["Bid"].shift(1))
    )

    flags = {
        "no_previous_record": ~has_previous,
        "invalid_adjacent_timestamps": has_previous & ~valid_pair,
        "positive_delta": positive,
        "zero_delta": equal,
        "backward_delta": backward,
        "source_boundary": has_previous & (
            joined["source_file"].ne(joined["source_file"].shift(1))
        ),
        "timestamp_parse_failed": failed,
        "timestamp_outside_supported_range": outside,
        "timestamp_clock_unresolved": unresolved,
        "invalid_price": ~good_price,
        "invalid_size": ~good_size,
        "crossed": good_price & numbers["Ask"].lt(numbers["Bid"]),
        "locked": good_price & numbers["Ask"].eq(numbers["Bid"]),
        "exact_raw_repeat_adjacent": has_previous & exact_raw,
        "equal_timestamp_changed_fields": equal & ~exact_raw,
        "same_bid_ask_later": positive & same_prices,
        "same_raw_quote_new_time": positive & same_raw_quote,
        "one_to_two_second_same_prices": (
            positive & same_prices
            & (delta_ns >= 1_000_000_000)
            & (delta_ns <= 2_000_000_000)
        ),
    }

    for column in VALUES:
        blank = joined[column].str.strip().eq("")
        flags[column + "_blank"] = blank
        flags[column + "_unparseable"] = ~blank & numbers[column].isna()
        flags[column + "_infinite"] = np.isinf(numbers[column])
        if "Volume" in column:
            flags[column + "_negative"] = numbers[column].lt(0)
            flags[column + "_zero"] = numbers[column].eq(0)
        else:
            flags[column + "_nonpositive"] = numbers[column].le(0)

    d["timestamp_athens"] = times
    d["previous_timestamp_athens"] = prior_times
    d["year"] = times.dt.year.fillna(0).astype("int32")
    d["gap_ns"] = delta_ns
    d["gap_seconds"] = np.where(
        valid_pair, delta_ns.astype("float64") / 1e9, np.nan
    )
    d["spread_pips"] = (
        (numbers["Ask"] - numbers["Bid"]) / PIP_SIZE
    ).where(good_price)

    closed = np.zeros(len(d), dtype=np.int64)
    before = closed.copy()
    after = closed.copy()
    middle = closed.copy()
    holiday = closed.copy()

    positions = np.flatnonzero(positive)
    if len(positions):
        a, b = prior_ns[positions], current_ns[positions]
        parts = overlap_parts(a, b, closures)
        for destination, values in zip(
            [closed, before, after, middle], parts
        ):
            destination[positions] = values
        holiday[positions] = overlap_parts(a, b, context)[0]

    covered = (
        valid_pair
        & (prior_ns >= bounds[0]) & (current_ns < bounds[1])
        & (current_ns >= bounds[0]) & (prior_ns < bounds[1])
    )
    population = np.full(len(d), "", dtype=object)
    population[positive] = "unannotated_open"
    population[positive & (holiday > 0)] = "holiday_context"
    population[positive & (closed > 0)] = "closure_crossing"
    population[positive & ~covered] = "calendar_uncovered"

    flags["quote_inside_expected_closure"] = points_inside(
        current_ns, times.notna().to_numpy(), closures
    )
    flags["timestamp_outside_calendar"] = (
        times.notna().to_numpy()
        & ((current_ns < bounds[0]) | (current_ns >= bounds[1]))
    )

    for name, values in flags.items():
        d[name] = np.asarray(values, dtype=bool)

    d["population"] = population
    d["closed_seconds"] = closed / 1e9
    d["open_before_seconds"] = before / 1e9
    d["open_after_seconds"] = after / 1e9
    d["open_middle_seconds"] = middle / 1e9

    d["bucket_index"] = -1
    d.loc[positive, "bucket_index"] = np.searchsorted(
        UPPER_NS, delta_ns[positive], side="left"
    )

    return d.iloc[offset:].reset_index(drop=True)
'''

In [ ]:
helper_sources["fx_audit_scan.py"] = r'''# Generated by Notebook 00
from collections import Counter, defaultdict
from pathlib import Path
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from fx_audit_rows import FIELDS, audit_rows

PAIR_COLUMNS = [
    "year", "source_file", "source_record",
    "previous_source_file", "previous_source_record",
    *FIELDS, *["previous_" + c for c in FIELDS],
    "timestamp_athens", "previous_timestamp_athens",
    "gap_ns", "gap_seconds", "population", "spread_pips",
    "closed_seconds", "open_before_seconds",
    "open_after_seconds", "open_middle_seconds",
]

CASES = [
    "timestamp_parse_failed",
    "timestamp_outside_supported_range",
    "timestamp_clock_unresolved",
    "backward_delta",
    "invalid_adjacent_timestamps",
    "exact_raw_repeat_adjacent",
    "equal_timestamp_changed_fields",
    "same_bid_ask_later",
    "same_raw_quote_new_time",
    "one_to_two_second_same_prices",
    "invalid_price", "invalid_size", "crossed", "locked",
    "quote_inside_expected_closure",
    "timestamp_outside_calendar",
]

LEDGER_COLUMNS = [
    "year", "source_file", "source_record",
    "previous_source_file", "previous_source_record",
    "timestamp_athens", "previous_timestamp_athens",
    "gap_ns", "gap_seconds", "population",
    "closed_seconds", "open_before_seconds",
    "open_after_seconds", "open_middle_seconds",
]

def scan_history(
    files, root, run_dir, expected_metadata,
    closures, context, bounds, chunk_rows, example_limit,
):
    run_dir, root = Path(run_dir), Path(root)
    ledger_path = run_dir / "gap_details.parquet"
    if ledger_path.exists() or (run_dir / "scan_complete.json").exists():
        raise RuntimeError(
            "This run already contains scan output. "
            "Rerun Cell 6 to create a new isolated run."
        )

    row_counts = defaultdict(Counter)
    ranges = {}
    hist_counts = Counter()
    hist_seconds = Counter()
    samples = {}
    tops = {}
    file_results = []
    previous = None
    writer = None
    ledger_schema = None
    ledger_rows = 0
    started = time.perf_counter()

    try:
        for path in files:
            relative = path.relative_to(root).as_posix()
            stat = path.stat()
            before_stat = (stat.st_size, stat.st_mtime_ns)
            if before_stat != expected_metadata[relative]:
                raise RuntimeError(f"Source changed before scanning: {relative}")

            count = 0
            file_first = None
            file_last = None
            print(f"Scanning {relative}", flush=True)

            reader = pd.read_csv(
                path, encoding="utf-8-sig", dtype="string",
                keep_default_na=False, na_filter=False,
                skip_blank_lines=False, on_bad_lines="error",
                chunksize=chunk_rows,
            )

            with reader:
                for raw in reader:
                    if list(raw.columns) != FIELDS:
                        raise ValueError(f"Schema changed: {relative}")

                    raw = raw.reset_index(drop=True)
                    raw["source_file"] = relative
                    raw["source_record"] = np.arange(
                        count + 1, count + len(raw) + 1, dtype=np.int64
                    )

                    d = audit_rows(
                        raw, previous, closures, context, bounds
                    )
                    previous = raw.tail(1).copy()
                    count += len(raw)

                    flag_columns = [
                        c for c in d.columns
                        if pd.api.types.is_bool_dtype(d[c])
                    ]

                    valid_times = d["timestamp_athens"].dropna()
                    if len(valid_times):
                        a, b = valid_times.min(), valid_times.max()
                        file_first = a if file_first is None else min(file_first, a)
                        file_last = b if file_last is None else max(file_last, b)

                    for year, block in d.groupby("year", sort=False):
                        year = int(year)
                        key = (relative, year)
                        target = row_counts[key]
                        target["rows"] += len(block)
                        for flag in flag_columns:
                            target[flag] += int(block[flag].sum())

                        times = block["timestamp_athens"].dropna()
                        if len(times):
                            a, b = times.min(), times.max()
                            old = ranges.get(key)
                            ranges[key] = (
                                (a, b) if old is None
                                else (min(old[0], a), max(old[1], b))
                            )

                        spreads = block.loc[
                            np.isfinite(block["spread_pips"]), "spread_pips"
                        ]
                        target["spread_count"] += len(spreads)
                        target["spread_sum_pips"] += float(spreads.sum())

                        for case in CASES:
                            sample_key = (year, case)
                            existing = samples.get(sample_key)
                            remaining = example_limit - (
                                0 if existing is None else len(existing)
                            )
                            if remaining > 0:
                                selected = block.loc[
                                    block[case], PAIR_COLUMNS
                                ].head(remaining).copy()
                                if len(selected):
                                    selected.insert(0, "case", case)
                                    samples[sample_key] = (
                                        selected if existing is None
                                        else pd.concat(
                                            [existing, selected], ignore_index=True
                                        )
                                    )

                        for label, eligible, metric in [
                            (
                                "largest_spread",
                                np.isfinite(block["spread_pips"]),
                                "spread_pips",
                            ),
                            (
                                "longest_same_price_interval",
                                block["same_bid_ask_later"],
                                "gap_seconds",
                            ),
                        ]:
                            selected = block.loc[
                                eligible, PAIR_COLUMNS
                            ].nlargest(example_limit, metric)
                            if len(selected):
                                top_key = (year, label)
                                prior = tops.get(top_key)
                                joined = (
                                    selected if prior is None
                                    else pd.concat([prior, selected], ignore_index=True)
                                )
                                tops[top_key] = joined.nlargest(example_limit, metric)

                    positive = d.loc[d["positive_delta"]]
                    if len(positive):
                        for (year, population, bucket), block in positive.groupby(
                            ["year", "population", "bucket_index"], sort=False
                        ):
                            key = (int(year), str(population), int(bucket))
                            hist_counts[key] += len(block)
                            hist_seconds[key] += float(block["gap_seconds"].sum())

                        keep = (
                            positive["gap_ns"].gt(10_000_000_000)
                            | positive["population"].isin([
                                "closure_crossing", "calendar_uncovered"
                            ])
                        )
                        detail = positive.loc[keep, LEDGER_COLUMNS].copy()

                        if len(detail):
                            detail["previous_source_record"] = (
                                detail["previous_source_record"].astype("int64")
                            )
                            table = pa.Table.from_pandas(
                                detail, preserve_index=False
                            )
                            if writer is None:
                                ledger_schema = table.schema
                                writer = pq.ParquetWriter(
                                    ledger_path, ledger_schema, compression="zstd"
                                )
                            writer.write_table(table.cast(ledger_schema))
                            ledger_rows += len(detail)

                    if count % 1_000_000 < len(raw):
                        print(f"  {count:,} source records", flush=True)

            final_stat = path.stat()
            if (final_stat.st_size, final_stat.st_mtime_ns) != before_stat:
                raise RuntimeError(f"Source changed during scanning: {relative}")

            file_results.append({
                "source_file": relative,
                "bytes": before_stat[0],
                "rows": count,
                "first_timestamp": file_first,
                "last_timestamp": file_last,
            })
            print(f"Finished: {count:,} records", flush=True)
    finally:
        if writer is not None:
            writer.close()

    row_records = []
    for (source, year), counts in row_counts.items():
        a, b = ranges.get((source, year), (None, None))
        row_records.append({
            "source_file": source,
            "year": year,
            **dict(counts),
            "first_timestamp": a,
            "last_timestamp": b,
        })

    histogram = pd.DataFrame([
        {
            "year": year,
            "population": population,
            "bucket_index": bucket,
            "count": count,
            "elapsed_seconds": hist_seconds[(year, population, bucket)],
        }
        for (year, population, bucket), count in hist_counts.items()
    ])

    example_parts = list(samples.values())
    for (year, label), frame in tops.items():
        frame = frame.copy()
        frame.insert(0, "case", label)
        example_parts.append(frame)

    examples = (
        pd.concat(example_parts, ignore_index=True)
        if example_parts
        else pd.DataFrame(columns=["case", *PAIR_COLUMNS])
    )

    return {
        "file_metrics": pd.DataFrame(file_results),
        "row_metrics": pd.DataFrame(row_records).fillna({
            "spread_count": 0, "spread_sum_pips": 0
        }),
        "histogram": histogram,
        "examples": examples,
        "ledger_rows": ledger_rows,
        "seconds_to_scan": time.perf_counter() - started,
    }
'''

In [ ]:
for filename, source in helper_sources.items():
    path = FUNCTION_DIR / filename

    if path.exists():
        old = path.read_text(encoding="utf-8")
        if not old.startswith("# Generated by Notebook 00"):
            raise RuntimeError(
                f"Refusing to replace an unrelated helper: {filename}"
            )

    path.write_text(source, encoding="utf-8")

if str(FUNCTION_DIR) not in sys.path:
    sys.path.insert(0, str(FUNCTION_DIR))

for name in [
    "fx_audit_calendar", "fx_audit_rows", "fx_audit_scan"
]:
    sys.modules.pop(name, None)

import fx_audit_calendar as audit_calendar
import fx_audit_rows as audit_rows
import fx_audit_scan as audit_scan

print("Loaded freshly written audit helpers.")

In [ ]:
verified_path = REPO_ROOT / "docs/verified_quote_closures.csv"

if not verified_path.exists():
    pd.DataFrame(
        columns=["start", "end", "reason", "source_url"]
    ).to_csv(verified_path, index=False)

verified_closures = pd.read_csv(
    verified_path, dtype="string", keep_default_na=False
)

assert list(verified_closures.columns) == [
    "start", "end", "reason", "source_url"
]

(
    calendar_events,
    closure_definitions,
    closure_windows,
    context_windows,
    calendar_bounds,
) = audit_calendar.build_calendar(
    FIRST_NOMINAL_YEAR,
    LAST_NOMINAL_YEAR,
    verified_closures,
)

calendar_events.to_csv(RUN_DIR / "calendar_events.csv", index=False)
closure_definitions.to_csv(RUN_DIR / "closure_definitions.csv", index=False)
verified_closures.to_csv(
    RUN_DIR / "verified_quote_closures_used.csv", index=False
)

buckets = pd.DataFrame({
    "bucket_index": np.arange(len(audit_rows.LABELS)),
    "bucket": audit_rows.LABELS,
    "lower_seconds_exclusive": audit_rows.LOWER_SECONDS,
    "upper_seconds_inclusive": [
        *audit_rows.UPPER_SECONDS.tolist(), np.inf
    ],
})

assert len(buckets) == 56
buckets.to_csv(RUN_DIR / "gap_bucket_definitions.csv", index=False)

display(buckets)
display(
    calendar_events.loc[
        calendar_events["kind"].isin([
            "eur_settlement", "us_payment"
        ])
        & calendar_events["nominal_year"].between(
            FIRST_NOMINAL_YEAR, LAST_NOMINAL_YEAR
        )
    ].head(25)
)

print("Verified additional quote-closure intervals:", len(verified_closures))
print("Calendar status: context coverage defined; quote-holiday hours provisional.")

In [ ]:
# Every finite bucket edge is right-inclusive.
for index, upper in enumerate(audit_rows.UPPER_NS):
    assert np.searchsorted(
        audit_rows.UPPER_NS, upper, side="left"
    ) == index

    assert np.searchsorted(
        audit_rows.UPPER_NS, upper + 1, side="left"
    ) == index + 1

assert audit_rows.LABELS[3] == "(5,10]"
assert audit_rows.LABELS[-1] == ">1800"

# Athens spring clock jump: one elapsed second.
times, failed, outside, unresolved = audit_rows.parse_athens(
    pd.Series([
        "2026.03.29 02:59:59.000",
        "2026.03.29 04:00:00.000",
    ], dtype="string")
)
assert audit_rows.ns_values(times)[1] - audit_rows.ns_values(times)[0] == 1_000_000_000

# Nonexistent and ambiguous local-clock times remain unresolved.
times, _, _, unresolved = audit_rows.parse_athens(
    pd.Series([
        "2026.03.29 03:30:00.000",
        "2026.10.25 03:30:00.000",
    ], dtype="string")
)
assert times.isna().all() and unresolved.all()

synthetic = pd.DataFrame([
    ["2016.01.04 12:00:00.000", "1.1000", "1.0999", "1", "1"],
    ["2016.01.04 12:00:00.000", "1.1000", "1.0999", "1", "1"],
    ["2016.01.04 12:00:00.000", "1.1001", "1.0999", "1", "1"],
    ["2016.01.04 12:00:01.000", "1.10010", "1.09990", "2", "1"],
    ["2016.01.04 12:00:02.000", "1.10010", "1.09990", "2", "1"],
], columns=FIELDS, dtype="string")

synthetic["source_file"] = "synthetic"
synthetic["source_record"] = np.arange(1, 6)

checked = audit_rows.audit_rows(
    synthetic, None, closure_windows, context_windows, calendar_bounds
)

assert checked.loc[1, "exact_raw_repeat_adjacent"]
assert checked.loc[2, "equal_timestamp_changed_fields"]
assert checked.loc[3, "same_bid_ask_later"]
assert checked.loc[4, "same_raw_quote_new_time"]

new_year_2022 = calendar_events.loc[
    calendar_events["nominal_year"].eq(2022)
    & calendar_events["holiday"].eq("New Year's Day")
    & calendar_events["kind"].eq("us_public_observed"),
    "date",
]
assert new_year_2022.tolist() == ["2021-12-31"]

assert not (
    calendar_events["kind"].eq("us_payment")
    & calendar_events["date"].eq("2022-01-03")
).any()

assert not (
    calendar_events["kind"].eq("us_payment")
    & calendar_events["date"].eq("2021-06-18")
).any()

assert audit_calendar.easter(2020).isoformat() == "2020-04-12"
assert audit_calendar.easter(2026).isoformat() == "2026-04-05"

print("PASS: bucket edges, DST arithmetic, unresolved times, repetitions, calendars.")

In [ ]:
# ## Expensive cell — complete raw-history scan

# The next cell reads every original CSV completely.

# It does not restrict the scan to a month, year, or sample.
# Files are read sequentially in the inventory order.
# Filename order is not assumed to prove chronological order.

# Memory use is bounded by the batch size, calendar definitions,
# small aggregates, and capped examples.

# Outputs include:

# - Exact parsed-record counts for every source.
# - Actual first/last parseable timestamps.
# - Yearly quality and repetition counts.
# - Every agreed gap bucket.
# - Diagnostic details for intervals longer than 10 seconds and all
#   positive closure-crossing/calendar-uncovered intervals.

# Malformed CSV input stops the scan rather than being skipped.
# Unparseable timestamp values are counted and retained as audit problems.

# An interrupted run has no completion marker. Start a fresh run by
# rerunning Cell 6 before repeating the scan.

In [ ]:
audit = audit_scan.scan_history(
    files=raw_files,
    root=REPO_ROOT,
    run_dir=RUN_DIR,
    expected_metadata=expected_metadata,
    closures=closure_windows,
    context=context_windows,
    bounds=calendar_bounds,
    chunk_rows=CHUNK_ROWS,
    example_limit=EXAMPLES_PER_CASE_PER_YEAR,
)

print("Full scan returned.")
print("Elapsed minutes:", round(audit["seconds_to_scan"] / 60, 2))

In [ ]:
file_metrics = audit["file_metrics"]
row_metrics = audit["row_metrics"]
raw_histogram = audit["histogram"]
examples = audit["examples"]

if row_metrics.empty or file_metrics["rows"].sum() == 0:
    raise RuntimeError("No source records were scanned.")

numeric_columns = [
    c for c in row_metrics.columns
    if c not in [
        "source_file", "year", "first_timestamp", "last_timestamp"
    ]
]

year_metrics = (
    row_metrics.groupby("year", as_index=False)[numeric_columns].sum()
    .sort_values("year")
)

year_metrics["mean_spread_pips"] = (
    year_metrics["spread_sum_pips"]
    / year_metrics["spread_count"].replace(0, np.nan)
)

populations = [
    "unannotated_open",
    "holiday_context",
    "closure_crossing",
    "calendar_uncovered",
]

hist_years = sorted(
    row_metrics.loc[row_metrics["year"].ne(0), "year"].unique()
)

grid = pd.MultiIndex.from_product(
    [hist_years, populations, buckets["bucket_index"]],
    names=["year", "population", "bucket_index"],
)

year_histogram = (
    raw_histogram
    .set_index(["year", "population", "bucket_index"])
    .reindex(grid, fill_value=0)
    .reset_index()
    .merge(buckets, on="bucket_index", how="left", validate="many_to_one")
)

year_histogram["count"] = year_histogram["count"].astype("int64")
year_histogram = year_histogram.sort_values(
    ["year", "population", "bucket_index"]
)

denominator = year_histogram.groupby(
    ["year", "population"]
)["count"].transform("sum")

year_histogram["frequency_percent"] = (
    100 * year_histogram["count"] / denominator.replace(0, np.nan)
)
year_histogram["cumulative_percent"] = (
    100
    * year_histogram.groupby(["year", "population"])["count"].cumsum()
    / denominator.replace(0, np.nan)
)

overall_histogram = (
    year_histogram.groupby(
        ["population", "bucket_index"], as_index=False
    )[["count", "elapsed_seconds"]].sum()
    .merge(buckets, on="bucket_index", how="left", validate="many_to_one")
    .sort_values(["population", "bucket_index"])
)

denominator = overall_histogram.groupby(
    "population"
)["count"].transform("sum")

overall_histogram["frequency_percent"] = (
    100 * overall_histogram["count"] / denominator.replace(0, np.nan)
)
overall_histogram["cumulative_percent"] = (
    100
    * overall_histogram.groupby("population")["count"].cumsum()
    / denominator.replace(0, np.nan)
)

file_metrics.to_csv(RUN_DIR / "file_metrics.csv", index=False)
row_metrics.to_csv(RUN_DIR / "quality_by_file_and_year.csv", index=False)
year_metrics.to_csv(RUN_DIR / "quality_by_year.csv", index=False)
year_histogram.to_csv(RUN_DIR / "gap_histogram_by_year.csv", index=False)
overall_histogram.to_csv(RUN_DIR / "gap_histogram_overall.csv", index=False)
examples.to_csv(RUN_DIR / "paired_examples.csv", index=False)

display(file_metrics)
display(year_metrics)

display(
    overall_histogram.loc[
        overall_histogram["population"].eq("unannotated_open"),
        ["bucket", "count", "frequency_percent",
         "cumulative_percent", "elapsed_seconds"],
    ]
)

In [ ]:
totals = row_metrics[numeric_columns].sum()
total_rows = int(totals["rows"])

positive_count = int(totals["positive_delta"])
accounted_rows = int(
    totals["no_previous_record"]
    + totals["invalid_adjacent_timestamps"]
    + totals["backward_delta"]
    + totals["zero_delta"]
    + totals["positive_delta"]
)

expected_ledger_rows = int(
    overall_histogram.loc[
        overall_histogram["bucket_index"].ge(4)
        | overall_histogram["population"].isin([
            "closure_crossing", "calendar_uncovered"
        ]),
        "count",
    ].sum()
)

checks = {
    "all_inventory_files_scanned": (
        set(file_metrics["source_file"])
        == set(inventory["source_file"])
    ),
    "file_and_year_row_totals_match": (
        int(file_metrics["rows"].sum())
        == int(year_metrics["rows"].sum())
        == total_rows
    ),
    "every_record_has_adjacency_accounting": accounted_rows == total_rows,
    "one_initial_record": int(totals["no_previous_record"]) == 1,
    "positive_intervals_match_histogram": (
        int(overall_histogram["count"].sum()) == positive_count
    ),
    "ledger_selection_reconciles": (
        expected_ledger_rows == audit["ledger_rows"]
    ),
    "source_boundaries_reconcile": (
        int(totals["source_boundary"])
        == int(file_metrics["rows"].gt(0).sum()) - 1
    ),
}

display(pd.DataFrame(
    [{"check": name, "passed": passed} for name, passed in checks.items()]
))

if not all(checks.values()):
    raise AssertionError("Structural reconciliation failed. Return these outputs.")

overlaps = []
records = file_metrics.to_dict("records")

for i, first in enumerate(records):
    if first["first_timestamp"] is None:
        continue
    for second in records[i + 1:]:
        if second["first_timestamp"] is None:
            continue

        start = max(first["first_timestamp"], second["first_timestamp"])
        end = min(first["last_timestamp"], second["last_timestamp"])

        if start <= end:
            overlaps.append({
                "first_file": first["source_file"],
                "second_file": second["source_file"],
                "overlap_start": start,
                "overlap_end": end,
                "shared_endpoint_only": start == end,
            })

overlap_table = pd.DataFrame(overlaps, columns=[
    "first_file", "second_file",
    "overlap_start", "overlap_end", "shared_endpoint_only",
])
overlap_table.to_csv(RUN_DIR / "source_range_overlaps.csv", index=False)

review_findings = {
    "unparseable_timestamps": int(totals["timestamp_parse_failed"]),
    "unsupported_timestamp_range": int(totals["timestamp_outside_supported_range"]),
    "unresolved_local_clock_times": int(totals["timestamp_clock_unresolved"]),
    "backward_intervals": int(totals["backward_delta"]),
    "timestamps_outside_calendar": int(totals["timestamp_outside_calendar"]),
    "quotes_inside_expected_closures": int(totals["quote_inside_expected_closure"]),
    "source_range_overlaps_or_shared_endpoints": len(overlap_table),
}

display(pd.DataFrame(
    [{"finding": name, "count": value}
     for name, value in review_findings.items()]
))
display(overlap_table)

print("PASS: complete structural accounting.")
print("Audit findings and calendar interpretation still require review.")

In [ ]:
print("Paired repetition examples:")
repeat_cases = [
    "exact_raw_repeat_adjacent",
    "equal_timestamp_changed_fields",
    "same_bid_ask_later",
    "same_raw_quote_new_time",
    "one_to_two_second_same_prices",
    "longest_same_price_interval",
]

display(
    examples.loc[
        examples["case"].isin(repeat_cases),
        [
            "case", "source_file", "source_record",
            "previous_source_file", "previous_source_record",
            "previous_timestamp_athens", "timestamp_athens",
            "gap_seconds",
            "previous_Ask", "Ask", "previous_Bid", "Bid",
            "previous_AskVolume", "AskVolume",
            "previous_BidVolume", "BidVolume",
        ],
    ].head(40)
)

ledger_path = RUN_DIR / "gap_details.parquet"
largest_gaps = pd.DataFrame()
closure_boundaries = pd.DataFrame()

if ledger_path.exists():
    connection = duckdb.connect(config={
        "memory_limit": "256MB",
        "threads": 2,
        "temp_directory": str(RUN_DIR / "duckdb_tmp"),
    })

    try:
        connection.read_parquet(str(ledger_path)).create_view("details")

        observed_ledger_rows = connection.execute(
            "SELECT COUNT(*) FROM details"
        ).fetchone()[0]

        assert observed_ledger_rows == audit["ledger_rows"]

        largest_gaps = connection.execute("""
            SELECT *
            FROM details
            WHERE population IN ('unannotated_open', 'holiday_context')
            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY year, population
                ORDER BY gap_ns DESC, source_file, source_record
            ) <= 20
            ORDER BY year, population, gap_ns DESC
        """).fetchdf()

        closure_boundaries = connection.execute("""
            SELECT *
            FROM details
            WHERE population = 'closure_crossing'
            ORDER BY timestamp_athens
        """).fetchdf()

        # Small closure-boundary output, not the complete tick history.
        if len(closure_boundaries):
            nonclosed = (
                closure_boundaries["gap_seconds"]
                - closure_boundaries["closed_seconds"]
            )
            pieces = closure_boundaries[[
                "open_before_seconds",
                "open_after_seconds",
                "open_middle_seconds",
            ]].sum(axis=1)

            assert np.allclose(nonclosed, pieces, atol=1e-6, rtol=1e-10)

    finally:
        connection.close()

largest_gaps.to_csv(RUN_DIR / "largest_open_gaps.csv", index=False)
closure_boundaries.to_csv(RUN_DIR / "closure_boundary_intervals.csv", index=False)

print("Largest open/context gaps:")
display(largest_gaps.head(30))

print("Closure-boundary intervals — open portions are censored:")
display(closure_boundaries.head(15))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

for population in ["unannotated_open", "holiday_context"]:
    frame = overall_histogram.loc[
        overall_histogram["population"].eq(population)
    ].sort_values("bucket_index")

    if frame["count"].sum() == 0:
        continue

    axes[0].plot(
        frame["bucket_index"],
        frame["count"].replace(0, np.nan),
        marker=".",
        label=population,
    )
    axes[1].plot(
        frame["bucket_index"],
        frame["cumulative_percent"],
        marker=".",
        label=population,
    )

axes[0].set_yscale("log")
axes[0].set_ylabel("Interval count — logarithmic scale")
axes[0].set_title(
    "Full-history quote-arrival intervals — provisional calendar classification"
)
axes[1].set_ylabel("Cumulative frequency (%)")
axes[1].set_xlabel("Elapsed-time bucket, seconds")
axes[1].set_ylim(0, 101)

ticks = np.unique(np.r_[np.arange(0, len(buckets), 3), len(buckets) - 1])
axes[1].set_xticks(ticks)
axes[1].set_xticklabels(
    buckets.loc[ticks, "bucket"], rotation=75, ha="right"
)

for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()

fig.tight_layout()
figure_path = FIGURE_DIR / "gap_frequency_distribution.png"
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
decisions_text = """# Design decisions

| Decision | Rationale | Component | Status |
|---|---|---|---|
| Audit every original CSV | Establish complete input coverage | 00 | Agreed |
| Source and display timezone Europe/Athens | Preserve agreed source-clock convention | All timing | Agreed baseline; validate against observations |
| Use actual elapsed time | Avoid DST arithmetic errors | All timing | Agreed |
| Preserve repeated observations | Repetition is not automatically corruption | 00–01 | Agreed |
| Use 56 specified gap buckets | Inspect the distribution before choosing a cutoff | 00 | Agreed |
| Keep public, settlement, and quote-closure calendars distinct | Avoid false closure exclusions | 00–01 | Agreed |
| 24/31 December are open holiday-affected EURUSD sessions | Retain thin-session observations | 00–01 | Agreed baseline |
| Disruption threshold unset | Requires distribution review and a chronological reference period | 00–02 | Pending |
| No quote filtering or gap filling in audit | ETL choices require discussion | 00–01 | Agreed |
| Holiday context scope is EUR/US plus agreed year-end dates | Other regional and exceptional events may remain | 00 | Provisional |
"""

decision_path = REPO_ROOT / "docs/design_decisions.md"
if not decision_path.exists():
    decision_path.write_text(decisions_text, encoding="utf-8")
else:
    marker = "## Notebook 00 initial audit definition"
    existing = decision_path.read_text(encoding="utf-8")
    if marker not in existing:
        decision_path.write_text(
            existing.rstrip() + "\n\n" + marker + "\n\n"
            + decisions_text.split("\n", 2)[2],
            encoding="utf-8",
        )

state_text = f"""# Project state

## Completed and reviewed notebooks

None yet.

## Current stage

Notebook 00 full-history scan executed.
Structural reconciliation passed.
Human review and interpretation remain pending.

## Inputs

Original CSVs directly inside data/.
No original observations were changed by the audit.

## Outputs

reports/tables/data_audit/{RUN_ID}/
reports/figures/data_audit/{RUN_ID}/

## Outstanding questions

- Interpret quality and repeated-observation examples.
- Resolve ordering problems or source overlaps, if present.
- Review holiday context and precise quote-closure evidence.
- Confirm displayed-size denomination.
- Agree on a chronological reference period before selecting a gap threshold.

## Next step

Review Notebook 00 outputs before designing ETL.

## Invalidated outputs

No downstream features, models, or simulations exist yet.
"""

(REPO_ROOT / "docs/project_state.md").write_text(
    state_text, encoding="utf-8"
)

contribution_path = REPO_ROOT / "CONTRIBUTIONS.md"
entry = f"""
## Notebook 00 execution — {RUN_ID}

The full-history audit was executed and its structural accounting checks passed.
Calendar interpretation, quality findings, and research validation await
Raymond's review. No gap threshold was selected.

Reports: reports/tables/data_audit/{RUN_ID}/
"""

with contribution_path.open("a", encoding="utf-8") as handle:
    handle.write(entry)

print("Saved decision and execution records. Human review remains pending.")

In [ ]:
valid_first = pd.to_datetime(
    file_metrics["first_timestamp"], utc=True
).dropna()
valid_last = pd.to_datetime(
    file_metrics["last_timestamp"], utc=True
).dropna()

SUMMARY = {
    "run_id": RUN_ID,
    "mode": "full_history",
    "source_files": len(raw_files),
    "source_bytes": int(inventory["bytes"].sum()),
    "source_records": total_rows,
    "first_timestamp_athens": (
        valid_first.min().tz_convert(TZ).isoformat()
        if len(valid_first) else None
    ),
    "last_timestamp_athens": (
        valid_last.max().tz_convert(TZ).isoformat()
        if len(valid_last) else None
    ),
    "positive_intervals": positive_count,
    "zero_intervals": int(totals["zero_delta"]),
    "backward_intervals": int(totals["backward_delta"]),
    "unresolved_adjacent_pairs": int(totals["invalid_adjacent_timestamps"]),
    "adjacent_exact_raw_repeats": int(totals["exact_raw_repeat_adjacent"]),
    "same_timestamp_changed_fields": int(totals["equal_timestamp_changed_fields"]),
    "same_bid_ask_at_later_timestamp": int(totals["same_bid_ask_later"]),
    "gap_detail_rows": int(audit["ledger_rows"]),
    "bucket_count": len(buckets),
    "structural_checks": checks,
    "review_findings": review_findings,
    "git_status": GIT_STATUS,
    "scan_minutes": round(audit["seconds_to_scan"] / 60, 3),
    "gap_threshold_seconds": None,
    "calendar_status": (
        "Selected EUR/US context calendars applied. "
        "Exact historical holiday quote-closure coverage remains provisional."
    ),
    "human_review": "pending",
    "source_stability_check": "file size and modification time; not content hashing",
}

(RUN_DIR / "audit_summary.json").write_text(
    json.dumps(SUMMARY, indent=2), encoding="utf-8"
)
(RUN_DIR / "scan_complete.json").write_text(
    json.dumps({
        "run_id": RUN_ID,
        "scan_completed": True,
        "structural_checks_passed": True,
        "human_review": "pending",
    }, indent=2),
    encoding="utf-8",
)

print("NOTEBOOK_00_EXECUTION_SUMMARY")
print(json.dumps(SUMMARY, indent=2))
print("\nTables:", RUN_DIR)
print("Figure:", figure_path)
print("\nSTOP HERE: return the requested outputs before ETL.")

In [ ]:
#Objective	What the outputs show	Decision needed
#Investigate unexplained gaps	336 unannotated intervals exceed 30 minutes; 334 occur in 2016–2018.	Can missing observations be recovered, or must these remain documented source disruptions?

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

review_run = (
    Path(RUN_DIR)
    if "RUN_DIR" in globals()
    else Path(
        input("Existing audit-run folder: ").strip().strip('"')
    ).resolve()
)

connection = duckdb.connect(config={
    "memory_limit": "256MB",
    "threads": 2,
})

try:
    connection.read_parquet(
        str(review_run / "gap_details.parquet")
    ).create_view("gap_details")

    gap_examples = connection.execute("""
        SELECT
            year,
            epoch_ns(previous_timestamp_athens) AS start_ns,
            epoch_ns(timestamp_athens) AS end_ns,
            gap_ns,
            gap_seconds,
            previous_source_file,
            previous_source_record,
            source_file,
            source_record
        FROM gap_details
        WHERE population = 'unannotated_open'
          AND gap_ns > 1800000000000
          AND year IN (2017, 2018)
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY year
            ORDER BY gap_ns DESC, source_file, source_record
        ) = 1
        ORDER BY year
    """).fetchdf()

finally:
    connection.close()

assert gap_examples["year"].tolist() == [2017, 2018]

for column in ["start_ns", "end_ns", "gap_ns"]:
    assert pd.api.types.is_integer_dtype(gap_examples[column].dtype)
    assert gap_examples[column].notna().all()

assert (
    gap_examples["end_ns"] - gap_examples["start_ns"]
).eq(gap_examples["gap_ns"]).all()

gap_examples["last_quote_athens"] = pd.to_datetime(
    gap_examples["start_ns"], unit="ns", utc=True
).dt.tz_convert("Europe/Athens")

gap_examples["next_quote_athens"] = pd.to_datetime(
    gap_examples["end_ns"], unit="ns", utc=True
).dt.tz_convert("Europe/Athens")

gap_examples["elapsed"] = pd.to_timedelta(
    gap_examples["gap_ns"], unit="ns"
)

with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(gap_examples[[
        "year",
        "last_quote_athens",
        "next_quote_athens",
        "elapsed",
        "gap_seconds",
        "previous_source_file",
        "previous_source_record",
        "source_file",
        "source_record",
    ]])

In [ ]:
from pathlib import Path
from bisect import bisect_right

import duckdb
import pandas as pd

# SETTINGS
audit_dir = REPO_ROOT / "reports" / "tables" / "data_audit" / "20261029T002746_883835Z"

cutoff_ns = 180 * 1_000_000_000
observation_hours = 2
END_EXCLUSION_HOURS = 1

research_tz = "Europe/Athens"
session_start_tz = "America/New_York"
session_start_hour = 17

# Sunday–Thursday 17:00 NY starts the five FX trading days.
session_start_weekdays = {6, 0, 1, 2, 3}

# Months whose replacement exports we have already reviewed.
reviewed_by_year = {
    2016: [1, 2, 3, 4, 5, 7, 12],
    2017: range(1, 13),
    2018: [1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12],
    2019: [1, 3, 5, 7, 8, 9, 10, 11, 12],
    2020: range(1, 13),
    2021: range(1, 13),
}
reviewed_months = {
    f"{year:04d}-{month:02d}"
    for year, months in reviewed_by_year.items()
    for month in months
}

# Read every interval that could qualify before exclusions.
with duckdb.connect() as con:
    recovery_gaps = con.execute(
        """
        SELECT
            epoch_ns(previous_timestamp_athens) AS start_ns,
            epoch_ns(timestamp_athens) AS end_ns,
            gap_ns,
            closed_seconds,
            source_file,
            source_record
        FROM read_parquet(?)
        WHERE gap_ns > ?
        ORDER BY start_ns
        """,
        [str(audit_dir / "gap_details.parquet"), cutoff_ns],
    ).df()

assert not recovery_gaps.empty

first_utc = pd.Timestamp(
    int(recovery_gaps["start_ns"].min()), unit="ns", tz="UTC"
)
last_utc = pd.Timestamp(
    int(recovery_gaps["end_ns"].max()), unit="ns", tz="UTC"
)

# Original closure calendar: preserve separately for validation.
calendar_table = pd.read_csv(audit_dir / "closure_definitions.csv")
calendar_intervals = []

for row in calendar_table.itertuples(index=False):
    start, end = pd.Timestamp(row.start), pd.Timestamp(row.end)
    assert start.tzinfo is not None and end.tzinfo is not None
    calendar_intervals.append((
        start.tz_convert("UTC").value,
        end.tz_convert("UTC").value,
    ))

# Agreed January 1 holiday exclusion, defined in Athens time.
holiday_intervals = []

for year in range(
    first_utc.tz_convert(research_tz).year - 1,
    last_utc.tz_convert(research_tz).year + 2,
):
    start = pd.Timestamp(f"{year}-01-01", tz=research_tz)
    end = pd.Timestamp(f"{year}-01-02", tz=research_tz)
    holiday_intervals.append((
        start.tz_convert("UTC").value,
        end.tz_convert("UTC").value,
    ))

# Provisional first-two-hours exclusion.
first_day = (
    first_utc.tz_convert(session_start_tz)
    .tz_localize(None).normalize() - pd.Timedelta(days=2)
)
last_day = (
    last_utc.tz_convert(session_start_tz)
    .tz_localize(None).normalize() + pd.Timedelta(days=2)
)

observation_intervals = []

for day in pd.date_range(first_day, last_day, freq="D"):
    if day.dayofweek not in session_start_weekdays:
        continue

    # Construct the local wall time directly, including on DST Sundays.
    start = pd.Timestamp(
        year=day.year,
        month=day.month,
        day=day.day,
        hour=session_start_hour,
        tz=session_start_tz,
    )
    end = start + pd.Timedelta(hours=observation_hours)

    observation_intervals.append((
        start.tz_convert("UTC").value,
        end.tz_convert("UTC").value,
    ))

scenarios = {
    "audit_calendar_check": calendar_intervals,
    "before_2h": calendar_intervals + holiday_intervals,
    "after_2h": (
        calendar_intervals
        + holiday_intervals
        + observation_intervals
    ),
}

print("Inputs ready. Run Cell 25.")

In [ ]:
comparison_rows = []

for scenario, exclusions in scenarios.items():

    # Merge overlapping or touching exclusions.
    merged = []
    for start, end in sorted(exclusions):
        assert start < end

        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))

    exclusion_ends = [end for start, end in merged]

    for gap_id, gap in enumerate(
        recovery_gaps.itertuples(index=False)
    ):
        start, end = int(gap.start_ns), int(gap.end_ns)
        assert end - start == int(gap.gap_ns)

        cursor = start
        index = bisect_right(exclusion_ends, start)
        retained_parts = []

        while index < len(merged) and merged[index][0] < end:
            excluded_start, excluded_end = merged[index]

            if excluded_start > cursor:
                retained_parts.append((cursor, excluded_start))

            cursor = max(cursor, excluded_end)
            if cursor >= end:
                break

            index += 1

        if cursor < end:
            retained_parts.append((cursor, end))

        # Validate the saved calendar before applying scenario exclusions.
        if scenario == "audit_calendar_check":
            calculated_closed_ns = (
                end - start
                - sum(b - a for a, b in retained_parts)
            )
            recorded_closed_ns = round(
                float(gap.closed_seconds) * 1_000_000_000
            )
            assert abs(calculated_closed_ns - recorded_closed_ns) <= 1_000
            continue

        for part_start, part_end in retained_parts:
            duration_ns = part_end - part_start

            # Strictly greater than three minutes.
            if duration_ns <= cutoff_ns:
                continue

            first = pd.Timestamp(
                part_start, unit="ns", tz="UTC"
            ).tz_convert(research_tz)

            last = pd.Timestamp(
                part_end - 1, unit="ns", tz="UTC"
            ).tz_convert(research_tz)

            months = pd.period_range(
                first.tz_localize(None).to_period("M"),
                last.tz_localize(None).to_period("M"),
                freq="M",
            )

            for month in months:
                comparison_rows.append({
                    "year_month": str(month),
                    "scenario": scenario,
                    "gap_id": gap_id,
                    "retained_minutes": duration_ns / 60_000_000_000,
                })

recovery_detail = pd.DataFrame(comparison_rows)

assert not recovery_detail.empty

# Count distinct original quote intervals touching each month.
monthly_comparison = (
    recovery_detail
    .groupby(["year_month", "scenario"])["gap_id"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=["before_2h", "after_2h"], fill_value=0)
)

longest_after = (
    recovery_detail.loc[
        recovery_detail["scenario"].eq("after_2h")
    ]
    .groupby("year_month")["retained_minutes"]
    .max()
)

monthly_comparison["longest_after_minutes"] = (
    longest_after.reindex(monthly_comparison.index)
    .fillna(0)
    .round(3)
)

monthly_comparison = monthly_comparison.reset_index()
monthly_comparison.columns.name = None

monthly_comparison["export_review"] = [
    "already reviewed" if month in reviewed_months else "not yet reviewed"
    for month in monthly_comparison["year_month"]
]

assert (
    monthly_comparison["after_2h"]
    <= monthly_comparison["before_2h"]
).all()

print("MONTHLY_RECOVERY_COMPARISON — ORIGINAL AUDIT SNAPSHOT")
with pd.option_context("display.max_rows", None):
    display(monthly_comparison)

# Remaining download candidates under the two-hour scenario.
pending_downloads = monthly_comparison.loc[
    monthly_comparison["after_2h"].gt(0)
    & monthly_comparison["export_review"].eq("not yet reviewed")
]

print("\nADJUSTED DOWNLOAD MONTHS BY YEAR")

if pending_downloads.empty:
    print("No unreviewed months qualify under this scenario.")
else:
    for year, group in pending_downloads.groupby(
        pending_downloads["year_month"].str[:4]
    ):
        months = ", ".join(group["year_month"].str[-2:])
        print(f"{year}: {months}")

print(f"\nUnreviewed months selected: {len(pending_downloads)}")

In [ ]:
from pathlib import Path
import csv
import json
import sys
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as arrow_csv
import duckdb

ROOT = REPO_ROOT
AUDIT_DIR = (
    ROOT / "reports" / "tables" / "data_audit"
    / "20261029T002746_883835Z"
)
NEW_DIR = ROOT / "data" / "2017 2018 investigation"

CUTOFF_MINUTES = 5
CUTOFF_NS = CUTOFF_MINUTES * 60 * 1_000_000_000
OBSERVATION_HOURS = 2
RESEARCH_TZ = "Europe/Athens"
BLOCK_BYTES = 16 * 1024 * 1024

# Previously reviewed export coverage.
reviewed_by_year = {
    2016: [1, 2, 3, 4, 5, 7, 12],
    2017: range(1, 13),
    2018: [1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12],
    2019: [1, 3, 5, 7, 8, 9, 10, 11, 12],
    2020: range(1, 13),
    2021: range(1, 13),
    2022: [1, 12],
}
reviewed_months = {
    f"{year:04d}-{month:02d}"
    for year, months in reviewed_by_year.items()
    for month in months
}

new_files = sorted(NEW_DIR.glob("*.csv"))
assert new_files, f"No CSV exports found in {NEW_DIR}"

expected_header = [
    "Time (EET)", "Ask", "Bid", "AskVolume", "BidVolume"
]
snapshot = {}
inventory_rows = []

for path in new_files:
    assert path.name.startswith("EURUSD_Ticks_"), path.name

    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        assert next(csv.reader(handle)) == expected_header, path.name

    stat = path.stat()
    snapshot[path] = (stat.st_size, stat.st_mtime_ns)
    inventory_rows.append({
        "file": path.name,
        "size_MB": round(stat.st_size / 1_000_000, 2),
    })

audit_summary = json.loads(
    (AUDIT_DIR / "audit_summary.json").read_text(encoding="utf-8")
)
assert all(audit_summary["structural_checks"].values())
assert audit_summary["unresolved_adjacent_pairs"] == 0

audit_start_ns = pd.Timestamp(
    audit_summary["first_timestamp_athens"]
).value
audit_end_ns = pd.Timestamp(
    audit_summary["last_timestamp_athens"]
).value

# Reuse this project's existing calendar helpers.
helper_dir = ROOT / "functions" / "data_audit"
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

from fx_audit_calendar import (
    merge_windows,
    overlap_parts,
    points_inside,
)

if "recovery_db" in globals():
    recovery_db.close()

recovery_db = duckdb.connect(
    ":memory:",
    config={"threads": 2, "memory_limit": "1GB"},
)

display(pd.DataFrame(inventory_rows))
print(f"New exports selected: {len(new_files)}")
print(
    "CSV volume to scan:",
    round(sum(size for size, mtime in snapshot.values()) / 1e9, 3),
    "GB",
)

In [ ]:
original_gaps = recovery_db.execute(
    """
    SELECT
        epoch_ns(previous_timestamp_athens) AS start_ns,
        epoch_ns(timestamp_athens) AS end_ns,
        gap_ns,
        closed_seconds,
        source_file,
        source_record
    FROM read_parquet(?)
    WHERE gap_ns > ?
    ORDER BY start_ns, end_ns
    """,
    [str(AUDIT_DIR / "gap_details.parquet"), CUTOFF_NS],
).df()

assert not original_gaps.empty

original_gaps.insert(
    0, "gap_id",
    np.arange(len(original_gaps), dtype=np.int64),
)

starts = original_gaps["start_ns"].to_numpy(dtype=np.int64)
ends = original_gaps["end_ns"].to_numpy(dtype=np.int64)
gap_ids = original_gaps["gap_id"].to_numpy(dtype=np.int64)

assert np.all(
    ends - starts
    == original_gaps["gap_ns"].to_numpy(dtype=np.int64)
)
assert np.all(ends[:-1] <= starts[1:])

# Original closure calendar.
calendar_table = pd.read_csv(AUDIT_DIR / "closure_definitions.csv")
calendar_windows = merge_windows([
    (
        pd.Timestamp(row.start).tz_convert("UTC").value,
        pd.Timestamp(row.end).tz_convert("UTC").value,
    )
    for row in calendar_table.itertuples(index=False)
])

# Confirm the saved calendar still agrees with the original ledger.
closed_ns, _, _, _ = overlap_parts(starts, ends, calendar_windows)
np.testing.assert_allclose(
    closed_ns / 1e9,
    original_gaps["closed_seconds"].to_numpy(),
    rtol=0,
    atol=1e-6,
)

exclusion_windows = list(calendar_windows)

# January 1 is excluded in Athens time.
first_year = pd.Timestamp(
    audit_start_ns, unit="ns", tz="UTC"
).tz_convert(RESEARCH_TZ).year

last_year = pd.Timestamp(
    audit_end_ns, unit="ns", tz="UTC"
).tz_convert(RESEARCH_TZ).year

for year in range(first_year - 1, last_year + 2):
    exclusion_windows.append((
        pd.Timestamp(
            f"{year}-01-01", tz=RESEARCH_TZ
        ).tz_convert("UTC").value,
        pd.Timestamp(
            f"{year}-01-02", tz=RESEARCH_TZ
        ).tz_convert("UTC").value,
    ))

# First two hours after Sunday–Thursday 17:00 New York.
first_day = (
    pd.Timestamp(audit_start_ns, unit="ns", tz="UTC")
    .tz_convert("America/New_York")
    .tz_localize(None).normalize()
    - pd.Timedelta(days=2)
)
last_day = (
    pd.Timestamp(audit_end_ns, unit="ns", tz="UTC")
    .tz_convert("America/New_York")
    .tz_localize(None).normalize()
    + pd.Timedelta(days=2)
)

for day in pd.date_range(first_day, last_day, freq="D"):
    if day.dayofweek not in {6, 0, 1, 2, 3}:
        continue

    start = pd.Timestamp(
        year=day.year, month=day.month, day=day.day,
        hour=17, tz="America/New_York",
    )
    end = start + pd.Timedelta(hours=OBSERVATION_HOURS)

    exclusion_windows.append((
        start.tz_convert("UTC").value,
        end.tz_convert("UTC").value,
    ))

exclusion_windows = merge_windows(exclusion_windows)
exclusions_df = pd.DataFrame(
    exclusion_windows, columns=["start_ns", "end_ns"]
).astype("int64")

recovery_db.register("original_gaps", original_gaps)
recovery_db.register("excluded_windows", exclusions_df)

# Each original gap always retains its original bounding observations.
recovery_db.execute("""
    CREATE TEMP TABLE recovery_points AS
    SELECT gap_id, start_ns AS ts_ns FROM original_gaps
    UNION ALL
    SELECT gap_id, end_ns AS ts_ns FROM original_gaps
""")

print(f"Original raw intervals selected: {len(original_gaps):,}")
print("Exclusion windows prepared.")

In [ ]:
scan_started = time.perf_counter()
source_rows = []

for file_number, path in enumerate(new_files, start=1):
    file_rows = 0
    matched_rows = 0
    first_ns = None
    previous_ns = None

    reader = arrow_csv.open_csv(
        str(path),
        read_options=arrow_csv.ReadOptions(
            block_size=BLOCK_BYTES,
        ),
        convert_options=arrow_csv.ConvertOptions(
            include_columns=["Time (EET)"],
            column_types={"Time (EET)": pa.string()},
        ),
    )

    try:
        for batch in reader:
            column = batch.column(0)
            assert column.null_count == 0, (
                f"{path.name}: missing timestamp"
            )

            local_times = pd.DatetimeIndex(pd.to_datetime(
                column.to_pandas(),
                format="%Y.%m.%d %H:%M:%S.%f",
                errors="raise",
            ))
            assert not local_times.hasnans, path.name

            utc_times = (
                local_times
                .tz_localize(
                    RESEARCH_TZ,
                    ambiguous="raise",
                    nonexistent="raise",
                )
                .tz_convert("UTC")
                .as_unit("ns")
            )
            ns = utc_times.asi8

            if len(ns) == 0:
                continue

            assert np.all(ns[1:] >= ns[:-1]), (
                f"{path.name}: backward timestamps within a batch"
            )
            if previous_ns is not None:
                assert int(ns[0]) >= previous_ns, (
                    f"{path.name}: backward timestamp across batches"
                )

            assert int(ns[0]) >= audit_start_ns, (
                f"{path.name}: extends before the original audit coverage"
            )
            assert int(ns[-1]) <= audit_end_ns, (
                f"{path.name}: extends after the original audit coverage"
            )

            # January 1 and observation windows are reporting exclusions,
            # not assertions that quotes cannot exist there.
            in_closure = points_inside(
                ns,
                np.ones(len(ns), dtype=bool),
                calendar_windows,
            )
            assert not in_closure.any(), (
                f"{path.name}: quotes conflict with the saved closure calendar"
            )

            if first_ns is None:
                first_ns = int(ns[0])

            previous_ns = int(ns[-1])
            file_rows += len(ns)

            # Original candidate intervals do not overlap.
            positions = np.searchsorted(starts, ns, side="right") - 1
            safe_positions = np.maximum(positions, 0)

            inside = (
                (positions >= 0)
                & (ns > starts[safe_positions])
                & (ns < ends[safe_positions])
            )

            if inside.any():
                new_points = pd.DataFrame({
                    "gap_id": gap_ids[positions[inside]],
                    "ts_ns": ns[inside],
                })

                recovery_db.register("_new_points_batch", new_points)
                recovery_db.execute("""
                    INSERT INTO recovery_points
                    SELECT gap_id, ts_ns FROM _new_points_batch
                """)
                recovery_db.unregister("_new_points_batch")

                matched_rows += int(inside.sum())

    finally:
        reader.close()

    assert file_rows > 0, f"{path.name}: no observations"

    current_stat = path.stat()
    assert (
        current_stat.st_size,
        current_stat.st_mtime_ns,
    ) == snapshot[path], f"{path.name}: changed during processing"

    source_rows.append({
        "file": path.name,
        "rows_scanned": file_rows,
        "rows_inside_original_gaps": matched_rows,
        "first_timestamp_athens": pd.Timestamp(
            first_ns, unit="ns", tz="UTC"
        ).tz_convert(RESEARCH_TZ).isoformat(),
        "last_timestamp_athens": pd.Timestamp(
            previous_ns, unit="ns", tz="UTC"
        ).tz_convert(RESEARCH_TZ).isoformat(),
    })

    print(
        f"{file_number}/{len(new_files)}: {path.name} — "
        f"{file_rows:,} rows scanned"
    )

assert set(new_files) == set(NEW_DIR.glob("*.csv")), (
    "The investigation folder changed. Rerun Cells 24–28."
)

source_report = pd.DataFrame(source_rows)
scan_minutes = (time.perf_counter() - scan_started) / 60

print(f"\nNew-export scan completed in {scan_minutes:.2f} minutes.")

In [ ]:
# Hand-checkable validation:
# parent 0 has observations every 300 seconds: no qualifying gaps.
# parent 1 splits into two gaps >300 seconds: still one parent interval.
test_result = recovery_db.execute("""
    WITH points(gap_id, t) AS (
        VALUES
            (0, 0), (0, 300), (0, 600), (0, 900), (0, 1200),
            (1, 0), (1, 600), (1, 1800)
    ),
    ordered AS (
        SELECT
            gap_id,
            t,
            lead(t) OVER (
                PARTITION BY gap_id ORDER BY t
            ) AS next_t
        FROM points
    )
    SELECT count(*), count(DISTINCT gap_id)
    FROM ordered
    WHERE next_t - t > 300
""").fetchone()

assert test_result == (2, 1)

# Timestamp presence only: this does not deduplicate quote records for ETL.
recovery_db.execute("""
    CREATE OR REPLACE TEMP TABLE distinct_points AS
    SELECT DISTINCT gap_id, ts_ns
    FROM recovery_points
""")

unique_point_count = recovery_db.execute(
    "SELECT count(*) FROM distinct_points"
).fetchone()[0]

bad_bounds = recovery_db.execute("""
    SELECT count(*)
    FROM (
        SELECT
            gap_id,
            min(ts_ns) AS lo,
            max(ts_ns) AS hi
        FROM distinct_points
        GROUP BY gap_id
    ) p
    JOIN original_gaps g USING (gap_id)
    WHERE p.lo <> g.start_ns OR p.hi <> g.end_ns
""").fetchone()[0]

assert bad_bounds == 0
assert unique_point_count >= 2 * len(original_gaps)

recovery_db.execute("""
    CREATE OR REPLACE TEMP TABLE updated_intervals AS
    WITH ordered AS (
        SELECT
            gap_id,
            ts_ns AS start_ns,
            lead(ts_ns) OVER (
                PARTITION BY gap_id ORDER BY ts_ns
            ) AS end_ns
        FROM distinct_points
    )
    SELECT gap_id, start_ns, end_ns
    FROM ordered
    WHERE end_ns - start_ns > ?
""", [CUTOFF_NS])

recovery_db.execute("""
    CREATE OR REPLACE TEMP TABLE review_intervals AS
    SELECT
        row_number() OVER () AS interval_id,
        dataset,
        gap_id,
        start_ns,
        end_ns
    FROM (
        SELECT
            'original' AS dataset,
            gap_id, start_ns, end_ns
        FROM original_gaps

        UNION ALL

        SELECT
            'updated' AS dataset,
            gap_id, start_ns, end_ns
        FROM updated_intervals
    ) q
""")

# Split at every exclusion boundary, then retain eligible portions.
recovery_db.execute("""
    CREATE OR REPLACE TEMP TABLE retained_parts AS
    WITH boundaries AS (
        SELECT interval_id, start_ns AS p
        FROM review_intervals

        UNION

        SELECT interval_id, end_ns AS p
        FROM review_intervals

        UNION

        SELECT
            i.interval_id,
            greatest(i.start_ns, e.start_ns) AS p
        FROM review_intervals i
        JOIN excluded_windows e
          ON e.start_ns < i.end_ns
         AND e.end_ns > i.start_ns

        UNION

        SELECT
            i.interval_id,
            least(i.end_ns, e.end_ns) AS p
        FROM review_intervals i
        JOIN excluded_windows e
          ON e.start_ns < i.end_ns
         AND e.end_ns > i.start_ns
    ),
    pieces AS (
        SELECT
            interval_id,
            p AS start_ns,
            lead(p) OVER (
                PARTITION BY interval_id ORDER BY p
            ) AS end_ns
        FROM boundaries
    )
    SELECT
        i.dataset,
        i.gap_id,
        p.start_ns,
        p.end_ns
    FROM pieces p
    JOIN review_intervals i USING (interval_id)
    WHERE p.end_ns - p.start_ns > ?
      AND NOT EXISTS (
          SELECT 1
          FROM excluded_windows e
          WHERE p.start_ns >= e.start_ns
            AND p.start_ns < e.end_ns
      )
""", [CUTOFF_NS])

retained_detail = recovery_db.execute("""
    SELECT
        p.*,
        g.source_file,
        g.source_record
    FROM retained_parts p
    JOIN original_gaps g USING (gap_id)
    ORDER BY p.dataset, p.start_ns
""").df()

original_ids = set(
    retained_detail.loc[
        retained_detail["dataset"].eq("original"), "gap_id"
    ]
)
updated_ids = set(
    retained_detail.loc[
        retained_detail["dataset"].eq("updated"), "gap_id"
    ]
)

# Additional observations cannot create a newly qualifying parent interval.
assert updated_ids <= original_ids

print("Updated gap calculation completed.")

In [ ]:
month_rows = []

for row in retained_detail.itertuples(index=False):
    first = pd.Timestamp(
        int(row.start_ns), unit="ns", tz="UTC"
    ).tz_convert(RESEARCH_TZ)

    last = pd.Timestamp(
        int(row.end_ns) - 1, unit="ns", tz="UTC"
    ).tz_convert(RESEARCH_TZ)

    months = pd.period_range(
        first.tz_localize(None).to_period("M"),
        last.tz_localize(None).to_period("M"),
        freq="M",
    )

    for month in months:
        month_rows.append({
            "year_month": str(month),
            "dataset": row.dataset,
            "gap_id": row.gap_id,
            "remaining_minutes": (
                int(row.end_ns) - int(row.start_ns)
            ) / 60_000_000_000,
        })

month_detail = pd.DataFrame(month_rows)

if month_detail.empty:
    monthly_report = pd.DataFrame(columns=[
        "year_month",
        "original_audit",
        "with_new_exports",
        "cleared",
        "longest_remaining_minutes",
        "export_review",
    ])
else:
    counts = (
        month_detail
        .groupby(["year_month", "dataset"])["gap_id"]
        .nunique()
        .unstack(fill_value=0)
        .reindex(columns=["original", "updated"], fill_value=0)
    )

    longest = (
        month_detail.loc[month_detail["dataset"].eq("updated")]
        .groupby("year_month")["remaining_minutes"]
        .max()
    )

    counts["longest_remaining_minutes"] = (
        longest.reindex(counts.index).fillna(0).round(3)
    )

    monthly_report = counts.rename(columns={
        "original": "original_audit",
        "updated": "with_new_exports",
    }).reset_index()
    monthly_report.columns.name = None

    monthly_report["cleared"] = (
        monthly_report["original_audit"]
        - monthly_report["with_new_exports"]
    )
    assert monthly_report["cleared"].ge(0).all()

    monthly_report["export_review"] = [
        "already reviewed"
        if month in reviewed_months
        else "not yet reviewed"
        for month in monthly_report["year_month"]
    ]

    monthly_report = monthly_report[[
        "year_month",
        "original_audit",
        "with_new_exports",
        "cleared",
        "longest_remaining_minutes",
        "export_review",
    ]]

print("MONTHLY_RECOVERY_5MIN_WITH_NEW_EXPORTS")
with pd.option_context("display.max_rows", None):
    display(monthly_report)

pending_downloads = monthly_report.loc[
    monthly_report["with_new_exports"].gt(0)
    & monthly_report["export_review"].eq("not yet reviewed")
]

print("\nADJUSTED DOWNLOAD MONTHS BY YEAR")

if pending_downloads.empty:
    print("No unreviewed months qualify.")
else:
    for year, group in pending_downloads.groupby(
        pending_downloads["year_month"].str[:4]
    ):
        print(
            f"{year}: "
            + ", ".join(group["year_month"].str[-2:])
        )

review_summary = {
    "original_audit_run": audit_summary["run_id"],
    "gap_cutoff_seconds": CUTOFF_MINUTES * 60,
    "observation_window": "17:00–19:00 America/New_York, Sunday–Thursday",
    "january_1_excluded": True,
    "new_export_files_scanned": len(new_files),
    "new_export_rows_scanned": int(source_report["rows_scanned"].sum()),
    "unique_added_timestamps_inside_original_gaps": int(
        unique_point_count - 2 * len(original_gaps)
    ),
    "original_qualifying_parent_intervals": len(original_ids),
    "remaining_qualifying_parent_intervals": len(updated_ids),
    "parent_intervals_cleared": len(original_ids - updated_ids),
    "unreviewed_months_selected": len(pending_downloads),
    "scan_minutes": round(scan_minutes, 3),
}

print("\nRECOVERY_REVIEW_SUMMARY")
print(json.dumps(review_summary, indent=2))

recovery_db.close()

In [ ]:
from bisect import bisect_right
import pandas as pd

required_state = [
    "retained_detail", "snapshot", "NEW_DIR",
    "audit_start_ns", "audit_end_ns",
    "CUTOFF_NS", "OBSERVATION_HOURS",
    "RESEARCH_TZ", "reviewed_months",
]
missing = [name for name in required_state if name not in globals()]

if missing:
    raise RuntimeError(
        "Run existing Cells 24–28 unchanged first. "
        f"Missing notebook state: {missing}"
    )

assert CUTOFF_NS == 300_000_000_000
assert OBSERVATION_HOURS == 2
assert RESEARCH_TZ == "Europe/Athens"

current_snapshot = {
    path: (path.stat().st_size, path.stat().st_mtime_ns)
    for path in sorted(NEW_DIR.glob("*.csv"))
}

if current_snapshot != snapshot:
    raise RuntimeError(
        "The investigation files changed after the previous scan. "
        "Rerun existing Cells 24–28 unchanged, then run Cells 29–31."
    )

# Preserve the original result for comparison.
base_detail_last_hour = retained_detail.drop_duplicates(
    ["dataset", "gap_id", "start_ns", "end_ns"]
).reset_index(drop=True)

END_OF_DAY_HOURS = 1

first_day_last_hour = (
    pd.Timestamp(audit_start_ns, unit="ns", tz="UTC")
    .tz_convert("America/New_York")
    .tz_localize(None).normalize()
    - pd.Timedelta(days=2)
)
last_day_last_hour = (
    pd.Timestamp(audit_end_ns, unit="ns", tz="UTC")
    .tz_convert("America/New_York")
    .tz_localize(None).normalize()
    + pd.Timedelta(days=2)
)

last_hour_windows = []

for day in pd.date_range(
    first_day_last_hour, last_day_last_hour, freq="D"
):
    # Monday–Friday includes the final hour before weekly closure.
    if day.dayofweek > 4:
        continue

    boundary = pd.Timestamp(
        year=day.year,
        month=day.month,
        day=day.day,
        hour=17,
        tz="America/New_York",
    )
    start = boundary - pd.Timedelta(hours=END_OF_DAY_HOURS)

    last_hour_windows.append((
        start.tz_convert("UTC").value,
        boundary.tz_convert("UTC").value,
    ))

last_hour_ends = [end for start, end in last_hour_windows]

# September–December 2023 has now been reviewed.
reviewed_months_last_hour = set(reviewed_months) | {
    "2023-09", "2023-10", "2023-11", "2023-12"
}

print("Inputs current.")
print("Additional exclusion: 16:00–17:00 New York, Monday–Friday.")

In [ ]:
last_hour_rows = []

for row in base_detail_last_hour.itertuples(index=False):
    start, end = int(row.start_ns), int(row.end_ns)
    cursor = start
    index = bisect_right(last_hour_ends, start)
    parts = []

    while (
        index < len(last_hour_windows)
        and last_hour_windows[index][0] < end
    ):
        excluded_start, excluded_end = last_hour_windows[index]

        if excluded_start > cursor:
            parts.append((cursor, excluded_start))

        cursor = max(cursor, excluded_end)
        if cursor >= end:
            break

        index += 1

    if cursor < end:
        parts.append((cursor, end))

    for part_start, part_end in parts:
        if part_end - part_start <= CUTOFF_NS:
            continue

        record = row._asdict()
        record["start_ns"] = part_start
        record["end_ns"] = part_end
        last_hour_rows.append(record)

retained_detail_last_hour = pd.DataFrame(
    last_hour_rows,
    columns=base_detail_last_hour.columns,
)

before_parent_keys = set(zip(
    base_detail_last_hour["dataset"],
    base_detail_last_hour["gap_id"],
))
after_parent_keys = set(zip(
    retained_detail_last_hour["dataset"],
    retained_detail_last_hour["gap_id"],
))

assert after_parent_keys <= before_parent_keys

policy_comparison_last_hour = pd.concat(
    [
        base_detail_last_hour.groupby("dataset")["gap_id"]
        .nunique().rename("before_last_hour_exclusion"),

        retained_detail_last_hour.groupby("dataset")["gap_id"]
        .nunique().rename("after_last_hour_exclusion"),
    ],
    axis=1,
).fillna(0).astype("int64")

assert (
    policy_comparison_last_hour["after_last_hour_exclusion"]
    <= policy_comparison_last_hour["before_last_hour_exclusion"]
).all()

print("EFFECT OF ADDING THE LAST-HOUR EXCLUSION")
display(policy_comparison_last_hour)

In [ ]:
last_hour_month_rows = []

for row in retained_detail_last_hour.itertuples(index=False):
    first = pd.Timestamp(
        int(row.start_ns), unit="ns", tz="UTC"
    ).tz_convert(RESEARCH_TZ)

    last = pd.Timestamp(
        int(row.end_ns) - 1, unit="ns", tz="UTC"
    ).tz_convert(RESEARCH_TZ)

    months = pd.period_range(
        first.tz_localize(None).to_period("M"),
        last.tz_localize(None).to_period("M"),
        freq="M",
    )

    for month in months:
        last_hour_month_rows.append({
            "year_month": str(month),
            "dataset": row.dataset,
            "gap_id": row.gap_id,
            "remaining_minutes": (
                int(row.end_ns) - int(row.start_ns)
            ) / 60_000_000_000,
        })

last_hour_month_detail = pd.DataFrame(last_hour_month_rows)

if last_hour_month_detail.empty:
    monthly_report_last_hour = pd.DataFrame(columns=[
        "year_month",
        "original_audit",
        "with_new_exports",
        "cleared",
        "longest_remaining_minutes",
        "export_review",
    ])
else:
    counts = (
        last_hour_month_detail
        .groupby(["year_month", "dataset"])["gap_id"]
        .nunique()
        .unstack(fill_value=0)
        .reindex(columns=["original", "updated"], fill_value=0)
    )

    longest = (
        last_hour_month_detail.loc[
            last_hour_month_detail["dataset"].eq("updated")
        ]
        .groupby("year_month")["remaining_minutes"]
        .max()
    )

    counts["longest_remaining_minutes"] = (
        longest.reindex(counts.index).fillna(0).round(3)
    )

    monthly_report_last_hour = counts.rename(columns={
        "original": "original_audit",
        "updated": "with_new_exports",
    }).reset_index()
    monthly_report_last_hour.columns.name = None

    monthly_report_last_hour["cleared"] = (
        monthly_report_last_hour["original_audit"]
        - monthly_report_last_hour["with_new_exports"]
    )
    assert monthly_report_last_hour["cleared"].ge(0).all()

    monthly_report_last_hour["export_review"] = [
        "already reviewed"
        if month in reviewed_months_last_hour
        else "not yet reviewed"
        for month in monthly_report_last_hour["year_month"]
    ]

    monthly_report_last_hour = monthly_report_last_hour[[
        "year_month",
        "original_audit",
        "with_new_exports",
        "cleared",
        "longest_remaining_minutes",
        "export_review",
    ]]

print("MONTHLY_RECOVERY_5MIN_WITH_NEW_EXPORTS")
print("Exclusions: first two hours, final hour, January 1 and closures.")

with pd.option_context("display.max_rows", None):
    display(monthly_report_last_hour)

pending_last_hour = monthly_report_last_hour.loc[
    monthly_report_last_hour["with_new_exports"].gt(0)
    & monthly_report_last_hour["export_review"].eq("not yet reviewed")
]

print("\nADJUSTED DOWNLOAD MONTHS BY YEAR")

if pending_last_hour.empty:
    print("No unreviewed months qualify.")
else:
    for year, group in pending_last_hour.groupby(
        pending_last_hour["year_month"].str[:4]
    ):
        print(
            f"{year}: "
            + ", ".join(group["year_month"].str[-2:])
        )

print(f"\nUnreviewed months selected: {len(pending_last_hour)}")